# Sparkify — Data Modeling with Apache Cassandra

**Author:** Moses Bargue Kortu Jr.  
**Stack:** Python 3 · Apache Cassandra · CQL · `cassandra-driver` · pandas

---

## Project context

Sparkify, a music streaming startup, collects user activity as a set of daily-partitioned CSV
event logs. The analytics team cannot query those raw files, and they need low-latency answers
to a small, fixed set of questions about listening behaviour.

This notebook delivers an end-to-end pipeline in two parts:

| Part | Deliverable |
|:--|:--|
| **I — ETL** | Consolidate the partitioned event logs in `event_data/` into a single denormalized file, `event_datafile_new.csv`. |
| **II — Data model** | Design, populate, and validate three Apache Cassandra tables, each purpose-built for one analytical query. |

## Modeling philosophy: the query comes first

Apache Cassandra is a distributed, partitioned row store. It deliberately omits the features a
relational engine relies on:

- **No `JOIN`s** — data cannot be assembled from multiple tables at read time.
- **No ad-hoc `WHERE`** — a predicate is only legal against partition and clustering key columns.
- **No query planner** — there is no optimizer to rescue a poorly shaped table.

The consequence is that *the schema is the query*. We do not model Sparkify's entities and then
ask questions of them; we start from each question, derive the `SELECT` that answers it, and
build a table whose primary key makes that `SELECT` a single-partition read. The same source
event is therefore written into all three tables. **That duplication is the design, not a
defect** — Cassandra trades storage, which is cheap, for read latency at scale, which is not.

Each section in Part II follows the same structure:

> **Requirement** → **Access pattern** → **Schema decision & rationale** → **`CREATE`** → **Load** → **`SELECT` validation**

---
# Part I — ETL: Pre-Processing the Event Files

### 1.1 Import dependencies

`cassandra.concurrent` is imported alongside the driver: it provides the bounded-concurrency
executor used in Part II to load roughly 6,800 rows into each table efficiently.

In [1]:
import csv
import glob
import os

import pandas as pd
from cassandra.cluster import Cluster
from cassandra.concurrent import execute_concurrent_with_args

pd.set_option("display.max_colwidth", 60)

### 1.2 Build the list of source event files

The raw logs are partitioned one file per day. We walk `event_data/`, collect every file path,
and sort the result so the consolidation step is **deterministic** — the same input directory
always produces a byte-identical output file, which makes the pipeline reproducible and
diff-able.

In [2]:
EVENT_DATA_DIR = os.path.join(os.getcwd(), "event_data")

file_path_list = []
for root, dirs, files in os.walk(EVENT_DATA_DIR):
    file_path_list.extend(glob.glob(os.path.join(root, "*.csv")))
file_path_list = sorted(file_path_list)

print("Source directory : {}".format(EVENT_DATA_DIR))
print("Files discovered : {}".format(len(file_path_list)))

Source directory : /workspace/home/event_data
Files discovered : 30


### 1.3 Consolidate the partitioned logs into a single denormalized file

Every source file is read and its rows accumulated, then written back out as one file. Two
transformations are applied:

1. **Column projection** — only the eleven fields required by the three analytical queries are
   retained; the remainder of the raw event schema is discarded.
2. **Null-artist filtering** — rows with an empty `artist` represent non-playback events such as
   page views and auth actions. They carry no listening signal and would pollute every downstream
   table, so they are dropped at the ETL boundary rather than filtered at query time.

In [3]:
OUTPUT_FILE = "event_datafile_new.csv"

OUTPUT_COLUMNS = [
    "artist", "firstName", "gender", "itemInSession", "lastName", "length",
    "level", "location", "sessionId", "song", "userId",
]

# Positional indices of the retained fields within the raw event schema.
SOURCE_INDICES = [0, 2, 3, 4, 5, 6, 7, 8, 12, 13, 16]

full_data_rows_list = []
for filepath in file_path_list:
    with open(filepath, "r", encoding="utf8", newline="") as csvfile:
        csvreader = csv.reader(csvfile)
        next(csvreader)                       # discard the per-file header
        full_data_rows_list.extend(csvreader)

csv.register_dialect("myDialect", quoting=csv.QUOTE_ALL, skipinitialspace=True)

rows_written = 0
with open(OUTPUT_FILE, "w", encoding="utf8", newline="") as f:
    writer = csv.writer(f, dialect="myDialect")
    writer.writerow(OUTPUT_COLUMNS)
    for row in full_data_rows_list:
        if row[0] == "":                      # drop non-playback events
            continue
        writer.writerow([row[i] for i in SOURCE_INDICES])
        rows_written += 1

print("Raw event rows read       : {:,}".format(len(full_data_rows_list)))
print("Non-playback rows dropped : {:,}".format(len(full_data_rows_list) - rows_written))
print("Rows written to {} : {:,}".format(OUTPUT_FILE, rows_written))

Raw event rows read       : 8,056
Non-playback rows dropped : 1,236
Rows written to event_datafile_new.csv : 6,820


### 1.4 Validate the consolidated dataset

Before anything is loaded into Cassandra we confirm the shape of the output and inspect its
contents. This is the last point at which a data-quality problem is cheap to fix; once the events
have been fanned out across three denormalized tables, correcting them means reloading all three.

In [4]:
df = pd.read_csv(OUTPUT_FILE, encoding="utf8")

print("Shape (rows, columns) : {}".format(df.shape))
print("Null values present   : {}".format(int(df.isnull().sum().sum())))

Shape (rows, columns) : (6820, 11)
Null values present   : 0


In [ ]:
df.head()

**Schema of `event_datafile_new.csv`**

| # | Column | Type | Role in the data model |
|:--|:--|:--|:--|
| 0 | `artist` | text | Returned by Q1, Q2 |
| 1 | `firstName` | text | Returned by Q2, Q3 |
| 2 | `gender` | text | Retained for future analysis |
| 3 | `itemInSession` | int | Clustering key in Q1, Q2 |
| 4 | `lastName` | text | Returned by Q2, Q3 |
| 5 | `length` | float | Returned by Q1 |
| 6 | `level` | text | Retained for future analysis |
| 7 | `location` | text | Retained for future analysis |
| 8 | `sessionId` | int | Partition key in Q1; partition key component in Q2 |
| 9 | `song` | text | Returned by Q1, Q2; partition key in Q3 |
| 10 | `userId` | int | Partition key component in Q2; clustering key in Q3 |

---
# Part II — Apache Cassandra Data Model

### 2.1 Connect to the cluster

A `Cluster` with a contact point of `127.0.0.1` targets the local Cassandra instance. A `Session`
is the driver's connection pool and query executor; one is shared across the whole notebook.

In [5]:
cluster = Cluster(["127.0.0.1"])
session = cluster.connect()

print("Connected to cluster.")

Connected to cluster.


### 2.2 Create and select the keyspace

A keyspace is the top-level container that defines the replication strategy for the tables within
it.

- **`SimpleStrategy` with `replication_factor = 1`** is used here because the notebook targets a
  single-node development instance. It places replicas around the ring without regard to physical
  topology.
- **In production**, this keyspace would use `NetworkTopologyStrategy` with a replication factor
  of 3 per datacenter, so replicas are distributed across racks and availability zones and a
  quorum survives the loss of any single node.

`IF NOT EXISTS` makes the statement idempotent, so the notebook can be re-run top to bottom
without error.

In [6]:
session.execute("""
    CREATE KEYSPACE IF NOT EXISTS sparkify
    WITH REPLICATION = {'class': 'SimpleStrategy', 'replication_factor': 1}
""")

session.set_keyspace("sparkify")

print("Keyspace 'sparkify' is ready and selected.")

Keyspace 'sparkify' is ready and selected.


### 2.3 A reusable, production-grade loader

All three tables are populated from the same source file, so the write path is factored into a
single helper rather than repeated three times. Three deliberate engineering choices are worth
calling out:

**Prepared statements.** Each `INSERT` is prepared once and then bound roughly 6,800 times. A
string-interpolated statement would force the coordinator node to re-parse identical CQL on every
row. Preparing also passes values as typed parameters, which removes CQL-injection risk and
reduces payload size.

**Bounded concurrency instead of a serial loop.** `execute_concurrent_with_args` keeps a fixed
number of requests in flight, so the loader is never blocked waiting on a full round trip per row.
This is the idiomatic bulk-write pattern in the DataStax driver.

**No `BATCH`.** Multi-partition batches are a well-known Cassandra anti-pattern: they force one
coordinator to fan out writes it does not own, creating a bottleneck and GC pressure. Batches
exist for atomicity within a single partition, not for throughput. Individual concurrent writes
are the correct tool here.

The helper also **fails loudly** — any write error is raised rather than silently swallowed, so a
partial load can never be mistaken for a successful one.

In [7]:
def load_table(insert_cql, row_mapper, table_name, concurrency=100):
    """Populate a Cassandra table from event_datafile_new.csv.

    Parameters
    ----------
    insert_cql : str
        A parameterized INSERT statement using '?' placeholders.
    row_mapper : callable
        Maps a csv.DictReader row (dict) to a tuple of bound values, applying the
        type casts required by the target table's column definitions.
    table_name : str
        Target table, used for the confirmation message.
    concurrency : int
        Maximum number of in-flight requests.
    """
    prepared = session.prepare(insert_cql)

    with open(OUTPUT_FILE, encoding="utf8") as f:
        parameters = [row_mapper(row) for row in csv.DictReader(f)]

    results = execute_concurrent_with_args(
        session, prepared, parameters, concurrency=concurrency
    )

    failures = [outcome for succeeded, outcome in results if not succeeded]
    if failures:
        raise RuntimeError(
            "{} of {} writes to {} failed. First error: {}".format(
                len(failures), len(parameters), table_name, failures[0]
            )
        )

    print("{}: {:,} rows loaded successfully.".format(table_name, len(parameters)))


def run_query(select_cql):
    """Execute a SELECT and return the result set as a pandas DataFrame for
    readable, tabular presentation.
    """
    rows = list(session.execute(select_cql))
    return pd.DataFrame(rows) if rows else pd.DataFrame()

> **A note on row-count verification.** `SELECT COUNT(*)` is intentionally not used to confirm
> load volume. In Cassandra it is an unbounded, cluster-wide scan that will time out on a real
> dataset. The loader's returned count is the correct verification signal; correctness of the data
> itself is verified by the three targeted `SELECT` statements below.

---
## Query 1 — Song details for a specific item within a session

### Requirement

> Return the **artist**, **song title**, and **song length** from the music app history that was
> heard during `sessionId = 338` and `itemInSession = 4`.

### Access pattern

| Aspect | Value |
|:--|:--|
| **Data returned** | `artist`, `song`, `length` |
| **Filtered on** | `session_id`, `item_in_session` |
| **Cardinality** | Exactly one row |

Working backwards from the requirement, the statement that answers it must be:

```sql
SELECT artist, song, length
FROM <table>
WHERE session_id = 338 AND item_in_session = 4;
```

### Schema decision & rationale

**Table name — `song_details_by_session`.** Named for what it returns and what it is queried by,
following the Cassandra `<result>_by_<predicate>` convention. The name alone tells a future
engineer which access pattern the table serves.

**Primary key — `PRIMARY KEY ((session_id), item_in_session)`**

| Component | Column | Justification |
|:--|:--|:--|
| **Partition key** | `session_id` | Determines which node owns the row. Every listening event in a session is co-located on one node, so the query resolves to a **single-partition read** — the fastest access Cassandra offers. Session IDs are high-cardinality and evenly distributed, so no hot partition forms. |
| **Clustering key** | `item_in_session` | Uniquely identifies a row within its partition and sorts events in playback order. |

**Why both columns must be in the primary key.** The `WHERE` clause filters on `session_id` *and*
`item_in_session`. Cassandra only permits predicates against primary key columns. If
`item_in_session` were an ordinary column the query would be rejected and would require
`ALLOW FILTERING` — a full-partition scan that is unacceptable in production and excluded by the
project specification. Declaring it a clustering key makes the filter a legal, efficient key-range
lookup.

**Why `session_id` alone would be insufficient.** A session contains many played items. With only
`session_id` in the primary key, every event in a session would overwrite the previous one and the
partition would collapse to a single row. The clustering key preserves all of them.

#### 1a. Create the table

In [8]:
session.execute("""
    CREATE TABLE IF NOT EXISTS song_details_by_session (
        session_id      int,
        item_in_session int,
        artist          text,
        song            text,
        length          float,
        PRIMARY KEY ((session_id), item_in_session)
    )
""")

print("Table 'song_details_by_session' is ready.")

Table 'song_details_by_session' is ready.


#### 1b. Load the table

In [9]:
load_table(
    insert_cql="""
        INSERT INTO song_details_by_session (session_id, item_in_session, artist, song, length)
        VALUES (?, ?, ?, ?, ?)
    """,
    row_mapper=lambda r: (
        int(r["sessionId"]),
        int(r["itemInSession"]),
        r["artist"],
        r["song"],
        float(r["length"]),
    ),
    table_name="song_details_by_session",
)

song_details_by_session: 6,820 rows loaded successfully.


#### 1c. Validate

The `SELECT` below is the query the table was designed to serve. It touches exactly one partition
and one row.

In [10]:
result_1 = run_query("""
    SELECT artist, song, length
    FROM song_details_by_session
    WHERE session_id = 338 AND item_in_session = 4
""")

result_1.round({"length": 4})

,artist,song,length
0,Faithless,Music Matters (Mark Knight Dub),495.3073


**Result.** A single row is returned, confirming both that the data loaded correctly and that the
primary key resolves the requirement to one unambiguous record.

---
## Query 2 — A user's ordered listening history within one session

### Requirement

> Return the **artist**, **song** (sorted by `itemInSession`), and the **user's first and last
> name** for `userId = 10` and `sessionId = 182`.

### Access pattern

| Aspect | Value |
|:--|:--|
| **Data returned** | `artist`, `song`, `first_name`, `last_name` |
| **Filtered on** | `user_id`, `session_id` |
| **Ordered by** | `item_in_session` |
| **Cardinality** | Many rows — one per song played |

```sql
SELECT artist, song, first_name, last_name
FROM <table>
WHERE user_id = 10 AND session_id = 182;
```

This requirement introduces a constraint Query 1 did not have: **result ordering**. That is not a
presentation concern to be handled in Python — it must be guaranteed by the storage layer.

### Schema decision & rationale

**Table name — `song_playlist_by_user_session`.** The result is effectively the playlist a user
worked through in one sitting, retrieved by user and session.

**Primary key — `PRIMARY KEY ((user_id, session_id), item_in_session)`**

| Component | Column(s) | Justification |
|:--|:--|:--|
| **Composite partition key** | `user_id`, `session_id` | Both are required filters, so both must sit in the primary key. Combining them into a *composite partition key* means the two-column filter resolves to one partition on one node. |
| **Clustering key** | `item_in_session` | Guarantees uniqueness within the partition and physically stores rows in playback order, satisfying the "sorted by `itemInSession`" requirement at read time with no sort cost. |

**Why a composite partition key rather than partitioning on `user_id` alone.** If `user_id` were
the sole partition key, every session a heavy user ever recorded would land in one partition on one
node. Power users would create **hot partitions** — unbounded partition growth, uneven storage, and
a single node absorbing disproportionate read traffic. Hashing on `(user_id, session_id)` spreads
each user's sessions across the ring, keeps every partition naturally bounded by the length of one
session, and still serves the query in a single read.

**Why ordering is free here.** Because `item_in_session` is the clustering column, Cassandra writes
rows to disk already sorted within the partition. The `SELECT` returns them in playback order with
no `ORDER BY` clause and no post-processing.

> **Production refinement.** `first_name` and `last_name` are invariant within a
> `(user_id, session_id)` partition — the same person is behind every row. Declaring them `STATIC`
> would store each value once per partition instead of once per row, reducing storage and write
> amplification. They are left as regular columns here to keep the schema aligned with the project
> specification.

#### 2a. Create the table

In [11]:
session.execute("""
    CREATE TABLE IF NOT EXISTS song_playlist_by_user_session (
        user_id         int,
        session_id      int,
        item_in_session int,
        artist          text,
        song            text,
        first_name      text,
        last_name       text,
        PRIMARY KEY ((user_id, session_id), item_in_session)
    )
""")

print("Table 'song_playlist_by_user_session' is ready.")

Table 'song_playlist_by_user_session' is ready.


#### 2b. Load the table

In [12]:
load_table(
    insert_cql="""
        INSERT INTO song_playlist_by_user_session (
            user_id, session_id, item_in_session, artist, song, first_name, last_name
        )
        VALUES (?, ?, ?, ?, ?, ?, ?)
    """,
    row_mapper=lambda r: (
        int(r["userId"]),
        int(r["sessionId"]),
        int(r["itemInSession"]),
        r["artist"],
        r["song"],
        r["firstName"],
        r["lastName"],
    ),
    table_name="song_playlist_by_user_session",
)

song_playlist_by_user_session: 6,820 rows loaded successfully.


#### 2c. Validate

No `ORDER BY` clause appears below. The rows come back in `item_in_session` order because the
clustering key already stores them that way.

In [13]:
result_2 = run_query("""
    SELECT artist, song, first_name, last_name
    FROM song_playlist_by_user_session
    WHERE user_id = 10 AND session_id = 182
""")

result_2

,artist,song,first_name,last_name
0,Down To The Bone,Keep On Keepin' On,Sylvie,Cruz
1,Three Drives,Greece 2000,Sylvie,Cruz
2,Sebastien Tellier,Kilometer,Sylvie,Cruz
3,Lonnie Gordon,Catch You Baby (Steve Pitron & Max Sanna Radio Edit),Sylvie,Cruz


**Result.** Four rows are returned in playback order, each attributed to the correct user —
confirming that the composite partition key and the clustering key both behave as designed.

---
## Query 3 — Every user who listened to a given song

### Requirement

> Return **every user's first and last name** in the music app history who listened to the song
> `'All Hands Against His Own'`.

### Access pattern

| Aspect | Value |
|:--|:--|
| **Data returned** | `first_name`, `last_name` |
| **Filtered on** | `song` |
| **Cardinality** | Many rows — one per distinct listener |

```sql
SELECT first_name, last_name
FROM <table>
WHERE song = 'All Hands Against His Own';
```

This query inverts the direction of the previous two. Queries 1 and 2 start from a session and ask
what was played; this one starts from a song and asks who played it. Cassandra cannot reach that
answer from either existing table — there is no `JOIN`, and `song` is not a key column in them — so
it requires a third table over the same source data. This is exactly the trade-off that query-first
modeling makes explicit.

### Schema decision & rationale

**Table name — `users_by_song`.** Returns users, queried by song.

**Primary key — `PRIMARY KEY ((song), user_id)`**

| Component | Column | Justification |
|:--|:--|:--|
| **Partition key** | `song` | The only filter in the requirement. Co-locates the entire listener list for a track in one partition, making the query a single-partition read. |
| **Clustering key** | `user_id` | Makes each `(song, user_id)` pair unique, so distinct listeners occupy distinct rows. |

**Why `user_id` is indispensable — the overwrite trap.** In Cassandra an `INSERT` is an upsert:
writing a row whose primary key already exists silently replaces it. If `song` were the entire
primary key, all ~6,800 events would collapse into one row per song and the query would return a
single arbitrary listener — whichever was written last. Adding `user_id` as a clustering key makes
the key unique per listener and preserves every one of them.

**A useful side effect.** The same upsert semantics that would have caused data loss now provide
free deduplication in the other direction: a user who plays the track five times still produces
exactly one row, which is precisely what "every user name" requires. The key is doing two jobs —
preventing loss across users, and collapsing repeats within a user.

> **Scaling caveat.** Partitioning on `song` means a viral track's partition grows with its
> listener count and could eventually become unbounded. At Sparkify's real scale the key would be
> bucketed — for example `((song, year_month), user_id)` — to cap partition size. The simple form is
> correct for this dataset; the limit is noted here to show it is understood rather than overlooked.

#### 3a. Create the table

In [14]:
session.execute("""
    CREATE TABLE IF NOT EXISTS users_by_song (
        song       text,
        user_id    int,
        first_name text,
        last_name  text,
        PRIMARY KEY ((song), user_id)
    )
""")

print("Table 'users_by_song' is ready.")

Table 'users_by_song' is ready.


#### 3b. Load the table

In [15]:
load_table(
    insert_cql="""
        INSERT INTO users_by_song (song, user_id, first_name, last_name)
        VALUES (?, ?, ?, ?)
    """,
    row_mapper=lambda r: (
        r["song"],
        int(r["userId"]),
        r["firstName"],
        r["lastName"],
    ),
    table_name="users_by_song",
)

users_by_song: 6,820 rows loaded successfully.


#### 3c. Validate

Each row below is a distinct listener. Repeat plays by the same user were deduplicated by the
primary key at write time.

In [16]:
result_3 = run_query("""
    SELECT first_name, last_name
    FROM users_by_song
    WHERE song = 'All Hands Against His Own'
""")

result_3

,first_name,last_name
0,Jacqueline,Lynch
1,Tegan,Levine
2,Sara,Johnson


**Result.** Three distinct listeners are returned, confirming that the clustering key preserved
every user rather than overwriting them.

---
# Part III — Teardown

### 3.1 Drop the tables

All three tables are dropped so the keyspace is returned to a clean state and the notebook can be
re-run from top to bottom without stale data. `IF EXISTS` keeps the statement idempotent.

In [17]:
for table in ("song_details_by_session", "song_playlist_by_user_session", "users_by_song"):
    session.execute("DROP TABLE IF EXISTS {}".format(table))
    print("Dropped {}.".format(table))

Dropped song_details_by_session.
Dropped song_playlist_by_user_session.
Dropped users_by_song.


### 3.2 Close the session and cluster connection

Shutting down releases the driver's connection pool and its background I/O threads. Leaving them
open leaks sockets and prevents the Python process from exiting cleanly.

In [18]:
session.shutdown()
cluster.shutdown()

print("Session and cluster connections closed.")

Session and cluster connections closed.


---
# Summary

### Delivered data model

| # | Business question | Table | Primary key | Read cost |
|:--|:--|:--|:--|:--|
| 1 | Song details for one item in a session | `song_details_by_session` | `((session_id), item_in_session)` | Single partition, one row |
| 2 | A user's ordered history in one session | `song_playlist_by_user_session` | `((user_id, session_id), item_in_session)` | Single partition, pre-sorted |
| 3 | Every listener of a given song | `users_by_song` | `((song), user_id)` | Single partition, deduplicated |

### Engineering decisions

- **Query-first modeling.** Each table was derived from its `SELECT`, not from Sparkify's entity
  relationships. Every query resolves to a single-partition read with no `ALLOW FILTERING`.
- **Composite partition keys prevent hot spots.** `(user_id, session_id)` distributes a heavy
  user's sessions across the ring instead of concentrating them on one node.
- **Clustering keys do double duty.** They order results at the storage layer (Q1, Q2) and
  guarantee row uniqueness so upserts do not destroy data (Q3).
- **Prepared statements with bounded concurrency.** Each `INSERT` is parsed once and bound thousands
  of times, executed with a fixed number of requests in flight — deliberately avoiding the
  multi-partition `BATCH` anti-pattern.
- **Failures surface immediately.** The loader raises on any write error, so a partial load can
  never be mistaken for a complete one.

### If this ran in production

The next steps would be `NetworkTopologyStrategy` replication at RF 3, bucketed partition keys to
bound the growth of `users_by_song`, `STATIC` columns for user attributes that repeat within a
partition, and idempotent re-runs driven by a scheduler rather than a notebook.